# Broken Chatbot ハンズオン — 00. セットアップ

LLM アプリ固有の脆弱性（**Prompt Leaking / P2SQL Injection / LLM4Shell**）を、実際に動く脆弱なチャットボットで攻撃・防御体験する講座です。

このノートブックでは環境構築と動作確認を行います。各脆弱性の演習は `10_prompt_leaking` / `20_p2sql` / `30_llm4shell` で行います。

---
### ⚠️ 安全に関する注意
- この教材は **意図的に脆弱** です。インターネットに公開しないでください。
- **LLM4Shell は実際にコードを実行（RCE）** します。**使い捨ての Colab ランタイム**での実行を推奨します。共有 JupyterHub や個人 PC では他者・自分の環境に影響し得ます。


## 0. 環境セットアップ

このノートブックを最初に一度実行してください。

下のセルを上から順に実行してください（Colab / JupyterHub / ローカル Jupyter 共通）。

**Step 1. リポジトリの取得とパス設定**

In [1]:
import os, sys, subprocess

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/13o-bbr-bbq/Broken_LLM_Integration_App.git"

def _find_root(start):
    d = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(d, "course")) and os.path.isdir(os.path.join(d, "chatapp")):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return None
        d = parent

if IN_COLAB:
    root = "/content/Broken_LLM_Integration_App"
    if not os.path.isdir(root):
        subprocess.run(["git", "clone", REPO_URL, root], check=True)
else:
    root = _find_root(os.getcwd())
    assert root, "repo root（course/ と chatapp/ を含む）が見つかりません"

os.chdir(root)
course_path = os.path.join(root, "course")
if course_path not in sys.path:
    sys.path.insert(0, course_path)
print("repo root:", root)

repo root: /Users/michiaki/workspace/03.LLMPJ/Broken_LLM_Integration_App


**Step 2. 依存パッケージのインストール**

In [ ]:
# 依存インストール（数分。Colab では新しいランタイムごとに必要）
!pip -q install -r course/requirements.txt

**Step 3. LLM プロバイダの設定**

OpenAI を使う場合は、事前に `course/.env` に API キーを記載しておくのが簡単です:
```bash
cp course/.env.example course/.env   # OPENAI_API_KEY を編集
```
`.env` が無い場合は Colab シークレット → 対話入力の順に解決されます。

In [2]:
from broken_handson import launcher

# LLM 設定を解決します（.env → 環境変数 → Colab シークレット → 対話入力 の順）。
#
# 【推奨】course/.env に OPENAI_API_KEY を書いておくと、このセルは何も尋ねずに完了します:
#     cp course/.env.example course/.env   # 作成後、OPENAI_API_KEY を編集
#
# Ollama を使う場合は course/.env に次を設定:
#     LLM_PROVIDER=ollama
#     OLLAMA_BASE_URL=http://<host>:11434
#     OLLAMA_MODEL_NAME=llama3.2
launcher.configure_llm()

[llm] provider=openai model=gpt-4o-mini（.env/環境変数の OPENAI_API_KEY を使用）


**Step 4. データベース（SQLite）の準備**

In [3]:
# SQLite を作成し、サンプルデータ（users 等）を投入（初回のみでOK）
launcher.seed()

2026-06-21 21:52:28,027 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-21 21:52:28,028 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("users")
2026-06-21 21:52:28,028 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-21 21:52:28,029 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("chats")
2026-06-21 21:52:28,029 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-21 21:52:28,029 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("messages")
2026-06-21 21:52:28,030 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-21 21:52:28,030 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("memberships")
2026-06-21 21:52:28,030 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-21 21:52:28,031 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("user_settings")
2026-06-21 21:52:28,031 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-21 21:52:28,031 INFO sqlalchemy.engine.Engine COMMIT
2026-06-21 21:52:28,036 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-21 21:5

'/Users/michiaki/workspace/03.LLMPJ/Broken_LLM_Integration_App/course/.runtime/broken_chatbot.db'

## 1. 動作確認（スモークテスト）

サーバを起動し、Web UI を表示します。

In [4]:
# uvicorn を --reload 付きで起動 → Web UI を表示
proc = launcher.start()
launcher.show()

[server] launching: /Users/michiaki/.pyenv/versions/3.12.9/bin/python3 -m uvicorn broken_handson.server:app --host 0.0.0.0 --port 8000 --reload --reload-dir /Users/michiaki/workspace/03.LLMPJ/Broken_LLM_Integration_App/chatapp/backend/app


INFO:     Will watch for changes in these directories: ['/Users/michiaki/workspace/03.LLMPJ/Broken_LLM_Integration_App/chatapp/backend/app']
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [65631] using WatchFiles
USER_AGENT environment variable not set, consider setting it to identify your requests.
/Users/michiaki/workspace/03.LLMPJ/Broken_LLM_Integration_App/chatapp/backend/app/llm_agent.py:8: LangChainDeprecationWarning: Importing WebBaseLoader from langchain.document_loaders is deprecated. Please replace deprecated imports:

>> from langchain.document_loaders import WebBaseLoader

with new imports of:

>> from langchain_community.document_loaders import WebBaseLoader
You can use the langchain cli to **automatically** upgrade many imports. Please see documentation here <https://python.langchain.com/v0.2/docs/versions/v0_2/>
  from langchain.document_loaders import WebBaseLoader
/Users/michiaki/.pyenv/versions/3.12.9/lib/pyt

2026-06-21 21:52:49,957 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-21 21:52:49,957 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("users")
2026-06-21 21:52:49,957 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-21 21:52:49,957 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("chats")
2026-06-21 21:52:49,957 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-21 21:52:49,958 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("messages")
2026-06-21 21:52:49,958 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-21 21:52:49,958 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("memberships")
2026-06-21 21:52:49,958 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-21 21:52:49,958 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("user_settings")
2026-06-21 21:52:49,958 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-21 21:52:49,958 INFO sqlalchemy.engine.Engine COMMIT
INFO:     127.0.0.1:63945 - "GET /healthcheck HTTP/1.1" 200 OK
[server] up: http://127

INFO:     Started server process [65633]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


INFO:     127.0.0.1:63947 - "GET / HTTP/1.1" 200 OK


UI 下部のドロップダウンでモードを選び、`Send a message` にプロンプトを入力して **Send** を押すと会話できます。

UI が表示されない環境では、次の `ask()` ヘルパで疎通確認できます。

In [ ]:
# UI を使わず Python から直接試すためのヘルパ（防御の自動検証にも便利）
import requests

def ask(mode, text, port=8000):
    r = requests.post(f"http://127.0.0.1:{port}/{mode}", json={"text": text}, timeout=120)
    return r.json().get("text")

print(ask('prompt-leaking-lv1', 'こんにちは。あなたは何の専門家ですか？'))

正しく応答が返ればセットアップ完了です。各脆弱性のノートブックへ進んでください。

確認が済んだらサーバを停止します（演習ノートブックで再度起動します）。

In [ ]:
# 後片付け（サーバ停止）
launcher.stop(proc)

---
### 次のステップ
- `10_prompt_leaking.ipynb` — システムプロンプト/APIキーの窃取と防御
- `20_p2sql.ipynb` — DB からのデータ窃取・改ざんと防御
- `30_llm4shell.ipynb` — リモートコード実行（RCE）と防御
